# Model Validation

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/antimicrobial-resistance=/path/to/antimicrobial-resistance"` at the top of any cell.

Visualise the simulation and inference outputs from the AMR two-strain colonisation model.

**Prerequisites:**
- Run `go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_simulation.yaml` to produce `dat/simulation_output.log`
- Run `go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_inference.yaml` to produce `dat/inference_output.log`

## Simulation Output

Load `dat/simulation_output.log` and plot the colonisation dynamics and infection process.

In [ ]:
import (
	"fmt"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

simStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/simulation_output.log")
if err != nil {
	panic(err)
}
fmt.Printf("Loaded simulation output: %d partitions\n", len(simStorage.GetNames()))

colonLine := analysis.NewLinePlotFromPartition(
	simStorage,
	analysis.DataRef{PartitionName: "colonisation", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "colonisation", ValueIndices: []int{0}},
		{PartitionName: "colonisation", ValueIndices: []int{1}},
	},
	nil,
)
colonLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Colonisation dynamics: susceptible & resistant fractions",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Fraction"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(colonLine, "width: 1024px; height: 400px; background: white;")

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

simStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/simulation_output.log")
if err != nil {
	panic(err)
}

infLine := analysis.NewLinePlotFromPartition(
	simStorage,
	analysis.DataRef{PartitionName: "infection", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "infection", ValueIndices: []int{0}},
		{PartitionName: "infection", ValueIndices: []int{1}},
	},
	nil,
)
infLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Infection process: susceptible & resistant BSI counts",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Count"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(infLine, "width: 1024px; height: 400px; background: white;")

## Inference Output

Load `dat/inference_output.log` and visualise posterior convergence for the four learned parameters:
`[transmission_rate, selection_coefficient, fitness_cost, community_resistant_prevalence]`

In [ ]:
import (
	"fmt"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

infStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/inference_output.log")
if err != nil {
	panic(err)
}
fmt.Printf("Loaded inference output: %d partitions\n", len(infStorage.GetNames()))

meanLine := analysis.NewLinePlotFromPartition(
	infStorage,
	analysis.DataRef{PartitionName: "params_posterior_mean", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "params_posterior_mean", ValueIndices: []int{0}},
		{PartitionName: "params_posterior_mean", ValueIndices: []int{1}},
		{PartitionName: "params_posterior_mean", ValueIndices: []int{2}},
		{PartitionName: "params_posterior_mean", ValueIndices: []int{3}},
	},
	nil,
)
meanLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Posterior mean convergence",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Parameter value"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Inference step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(meanLine, "width: 1024px; height: 450px; background: white;")

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

infStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/inference_output.log")
if err != nil {
	panic(err)
}

covLine := analysis.NewLinePlotFromPartition(
	infStorage,
	analysis.DataRef{PartitionName: "params_posterior_cov", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "params_posterior_cov", ValueIndices: []int{0}},
		{PartitionName: "params_posterior_cov", ValueIndices: []int{5}},
		{PartitionName: "params_posterior_cov", ValueIndices: []int{10}},
		{PartitionName: "params_posterior_cov", ValueIndices: []int{15}},
	},
	nil,
)
covLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Posterior variance convergence (diagonal elements)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Variance"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Inference step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(covLine, "width: 1024px; height: 450px; background: white;")

## Parameter Samples

Plot the sampled parameter values from the generating process to visualise the posterior distribution.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

infStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/inference_output.log")
if err != nil {
	panic(err)
}

paramScatter := analysis.NewScatterPlotFromPartition(
	infStorage,
	analysis.DataRef{PartitionName: "params_generating_process", ValueIndices: []int{0}},
	[]analysis.DataRef{
		{PartitionName: "params_generating_process", ValueIndices: []int{1}},
	},
)
paramScatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Parameter samples: transmission rate vs selection coefficient",
		Bottom: "1%",
	}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Transmission rate"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Selection coefficient"}),
)
gonb_echarts.Display(paramScatter, "width: 1024px; height: 500px; background: white;")

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

infStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/inference_output.log")
if err != nil {
	panic(err)
}

paramScatter2 := analysis.NewScatterPlotFromPartition(
	infStorage,
	analysis.DataRef{PartitionName: "params_generating_process", ValueIndices: []int{2}},
	[]analysis.DataRef{
		{PartitionName: "params_generating_process", ValueIndices: []int{3}},
	},
)
paramScatter2.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Parameter samples: fitness cost vs community resistant prevalence",
		Bottom: "1%",
	}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Fitness cost"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Community resistant prevalence"}),
)
gonb_echarts.Display(paramScatter2, "width: 1024px; height: 500px; background: white;")

## Log-Normalisation

Track the posterior log-normalisation over inference steps — this should stabilise as the posterior converges.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

infStorage, err := analysis.NewStateTimeStorageFromJsonLogEntries("../dat/inference_output.log")
if err != nil {
	panic(err)
}

normLine := analysis.NewLinePlotFromPartition(
	infStorage,
	analysis.DataRef{PartitionName: "params_posterior_log_norm", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "params_posterior_log_norm", ValueIndices: []int{0}},
	},
	nil,
)
normLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Posterior log-normalisation",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Log-normalisation"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Inference step"}),
)
gonb_echarts.Display(normLine, "width: 1024px; height: 400px; background: white;")